In [7]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import HumanMessage

# Ensure we're working from the project root
project_root = Path("/Users/nurma/vscode_projects/Agent_from_scratch")
os.chdir(project_root)
print(f"Current working directory: {os.getcwd()}")

load_dotenv()

model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)

# Use absolute path to the MCP server
mcp_server_path = project_root / "Test_task" / "MCP_test_task.py"
print(f"MCP server path: {mcp_server_path}")
print(f"MCP server exists: {mcp_server_path.exists()}")

# Use the same Python interpreter as the notebook (from venv)
python_path = sys.executable
print(f"Using Python: {python_path}")

client = MultiServerMCPClient(
    {
        "product_manager": {
            "command": python_path,
            "args": [str(mcp_server_path)],
            "transport": "stdio",
        },
    }
)

async def run_multi_server_agent():
    """Create and run agent with multiple MCP servers"""
    
    # IMPORTANT: Need to await async functions!
    print("Loading tools from MCP server...")
    tools = await client.get_tools()
    
    print(f"Loaded {len(tools)} tools")
    print("Available tools:", [tool.name for tool in tools])
    
    agent = create_agent(model, tools)
    
    # Also need to await ainvoke since it's async
    response = await agent.ainvoke({
        "messages": "get all the products"
    })
    
    print("\n=== Response ===")
    for message in response["messages"]:
        print(message)
    
    return response, tools

# In Jupyter notebooks, you can use await directly in cells (top-level await)
response, tools = await run_multi_server_agent()

Current working directory: /Users/nurma/vscode_projects/Agent_from_scratch
MCP server path: /Users/nurma/vscode_projects/Agent_from_scratch/Test_task/MCP_test_task.py
MCP server exists: True
Using Python: /Users/nurma/vscode_projects/Agent_from_scratch/.venv/bin/python
Loading tools from MCP server...
Loaded 4 tools
Available tools: ['get_all_products', 'add_new_product', 'get_product_by_id', 'get_statistics']

=== Response ===
content='get all the products' additional_kwargs={} response_metadata={} id='7dab805c-739c-46cf-af64-f2fb5481b16d'
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 143, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_159

In [13]:
tools[0].__dict__

{'name': 'get_all_products',
 'description': 'Возвращает список всех продуктов.',
 'args_schema': {'properties': {},
  'title': 'get_all_productsArguments',
  'type': 'object'},
 'return_direct': False,
 'verbose': False,
 'callbacks': None,
 'tags': None,
 'metadata': None,
 'handle_tool_error': False,
 'handle_validation_error': False,
 'response_format': 'content_and_artifact',
 'extras': None,
 'func': None,
 'coroutine': <function langchain_mcp_adapters.tools.convert_mcp_tool_to_langchain_tool.<locals>.call_tool(runtime: Any = None, **arguments: dict[str, typing.Any]) -> tuple[list[langchain_core.messages.content.TextContentBlock | langchain_core.messages.content.ImageContentBlock | langchain_core.messages.content.FileContentBlock] | langchain_core.messages.tool.ToolMessage | langgraph.types.Command, langchain_mcp_adapters.tools.MCPToolArtifact | None]>,
 '_injected_args_keys': frozenset()}

In [14]:
tools

[StructuredTool(name='get_all_products', description='Возвращает список всех продуктов.', args_schema={'properties': {}, 'title': 'get_all_productsArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x111361080>),
 StructuredTool(name='add_new_product', description='Добавляет новый продукт в базу. Returns the created product as a dictionary.', args_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'price': {'title': 'Price', 'type': 'number'}, 'category': {'title': 'Category', 'type': 'string'}, 'in_stock': {'title': 'In Stock', 'type': 'boolean'}}, 'required': ['name', 'price', 'category', 'in_stock'], 'title': 'add_new_productArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x111432700>),
 StructuredTool(name='get_product_by_id', description='Ищет продукт по ID. Если не найд

In [15]:
tools[1].__dict__

{'name': 'add_new_product',
 'description': 'Добавляет новый продукт в базу. Returns the created product as a dictionary.',
 'args_schema': {'properties': {'name': {'title': 'Name', 'type': 'string'},
   'price': {'title': 'Price', 'type': 'number'},
   'category': {'title': 'Category', 'type': 'string'},
   'in_stock': {'title': 'In Stock', 'type': 'boolean'}},
  'required': ['name', 'price', 'category', 'in_stock'],
  'title': 'add_new_productArguments',
  'type': 'object'},
 'return_direct': False,
 'verbose': False,
 'callbacks': None,
 'tags': None,
 'metadata': None,
 'handle_tool_error': False,
 'handle_validation_error': False,
 'response_format': 'content_and_artifact',
 'extras': None,
 'func': None,
 'coroutine': <function langchain_mcp_adapters.tools.convert_mcp_tool_to_langchain_tool.<locals>.call_tool(runtime: Any = None, **arguments: dict[str, typing.Any]) -> tuple[list[langchain_core.messages.content.TextContentBlock | langchain_core.messages.content.ImageContentBlock |

In [19]:
from langchain_core.tools import tool

@tool("add", description="takes sum of two numbers")
def add(num1: int, num2: int):
    return num1 + num2

add

StructuredTool(name='add', description='takes sum of two numbers', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x111789c60>)

In [20]:
tool()
def add(a: float, b: float) -> float:
    """Add two numbers together

    Args:
        a: First number
        b: Second number

    Returns:
        The sum of a and b
    """
    return a + b


tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers

    Args:
        a: First number
        b: Second number

    Returns:
        The product of a and b
    """
    return a * b


tool()
def divide(a: float, b: float) -> str:
    """Divide two numbers with zero check

    Args:
        a: Numerator
        b: Denominator

    Returns:
        Result of division or error message if dividing by zero
    """
    if b == 0:
        return "Error: Cannot divide by zero"
    result = a / b
    return f"{a} ÷ {b} = {result}"


tool()
def power(base: float, exponent: float) -> float:
    """Raise a number to a power

    Args:
        base: The base number
        exponent: The exponent

    Returns:
        base raised to the power of exponent
    """
    return base ** exponent
